In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
import xgboost as xgb

np.random.seed(42)

In [ ]:
df = pd.read_csv('corporate_ai_adoption_dataset.csv')
print(f"Shape: {df.shape}")
print(df.head())

# Check for missing values
print(f"\nMissing values:\n{df.isnull().sum()}")

# Drop rows with missing target values
df = df.dropna(subset=['ai_maturity_score', 'ai_adoption_level'])
print(f"Shape after dropping NaN: {df.shape}")

Shape: (200000, 13)
   company_id            industry        country  year  ai_adoption_level  \
0  CORP-06613  Financial Services          China  2029             0.4987   
1  CORP-01597         Agriculture        Germany  2032             0.5213   
2  CORP-02938              Energy  United States  2024             0.6147   
3  CORP-05207              Retail        Germany  2021             0.4401   
4  CORP-07489          Technology  United States  2024             0.1918   

   ai_investment_usd  automation_rate  cost_savings  revenue_impact  \
0           11747237           0.4119       3519354         2751513   
1            1267219           0.4580        295244          304776   
2            8168951           0.5821       2472329         5170304   
3            1234261           0.2880        512061         -233448   
4            5000645           0.1906       2122064         2110644   

   productivity_gain  employee_ai_training_hours  ai_maturity_score  \
0             0.392

In [ ]:
df_processed = df.copy()

# Encode categorical columns (industry, country)
le_dict = {}
cat_cols = ['industry', 'country']
for col in cat_cols:
    le = LabelEncoder()
    df_processed[col + '_enc'] = le.fit_transform(df_processed[col].astype(str))
    le_dict[col] = le
    print(f"{col}: {df_processed[col].nunique()} values encoded")

# Create classification target (Low/Medium/High) from ai_adoption_level
df_processed['adoption_category'] = pd.qcut(
    df_processed['ai_adoption_level'], q=3,
    labels=['Low_Adoption', 'Medium_Adoption', 'High_Adoption'],
    duplicates='drop'
)
le_adoption = LabelEncoder()
df_processed['adoption_category_enc'] = le_adoption.fit_transform(df_processed['adoption_category'])
le_dict['adoption_category'] = le_adoption
adoption_labels = le_adoption.classes_
print(f"\nAdoption classes: {adoption_labels}")
print(df_processed['adoption_category'].value_counts())

# Define features (drop ID and target columns)
DROP_COLS = ['company_id', 'industry', 'country',
             'ai_maturity_score',           # regression target
             'adoption_category',           # classification target (raw)
             'adoption_category_enc',       # classification target (encoded)
             'ai_adoption_level']           # used to create target - exclude to avoid leakage

feature_cols = [c for c in df_processed.columns if c not in DROP_COLS]
print(f"\nFeatures: {feature_cols}")

X = df_processed[feature_cols]
y_reg = df_processed['ai_maturity_score']       # regression target
y_clf = df_processed['adoption_category_enc']   # classification target

print(f"X shape: {X.shape}")
print(f"y_reg shape: {y_reg.shape}")
print(f"y_clf shape: {y_clf.shape}")


industry: 10 values encoded
country: 15 values encoded

Adoption classes: ['High_Adoption' 'Low_Adoption' 'Medium_Adoption']
adoption_category
Low_Adoption       66688
High_Adoption      66657
Medium_Adoption    66655
Name: count, dtype: int64

Features: ['year', 'ai_investment_usd', 'automation_rate', 'cost_savings', 'revenue_impact', 'productivity_gain', 'employee_ai_training_hours', 'deployment_count', 'industry_enc', 'country_enc']
X shape: (200000, 10)
y_reg shape: (200000,)
y_clf shape: (200000,)


In [ ]:
X_train, X_test, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)
_, _, y_train_clf, y_test_clf = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print(f"\nTrain: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")


Train: 160,000 | Test: 40,000


In [ ]:
print("\n" + "="*50)
print("TRAINING RANDOM FOREST")
print("="*50)

# Regression
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=15,
                                min_samples_split=10, n_jobs=-1, random_state=42)
rf_reg.fit(X_train, y_train_reg)
y_pred_rf_reg = rf_reg.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_rf_reg))
rf_r2 = r2_score(y_test_reg, y_pred_rf_reg)
print(f"Regression - RMSE: {rf_rmse:.4f}, R²: {rf_r2:.4f}")

# Classification
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=15,
                                 min_samples_split=10, n_jobs=-1, random_state=42)
rf_clf.fit(X_train, y_train_clf)
y_pred_rf_clf = rf_clf.predict(X_test)
rf_acc = accuracy_score(y_test_clf, y_pred_rf_clf)
print(f"Classification - Accuracy: {rf_acc:.4f}")


TRAINING RANDOM FOREST
Regression - RMSE: 0.2996, R²: 0.9759
Classification - Accuracy: 0.3323


In [ ]:
print("\n" + "="*50)
print("TRAINING GRADIENT BOOSTING")
print("="*50)

# Regression
gb_reg = GradientBoostingRegressor(n_estimators=150, learning_rate=0.05,
                                    max_depth=5, subsample=0.8, random_state=42)
gb_reg.fit(X_train, y_train_reg)
y_pred_gb_reg = gb_reg.predict(X_test)

gb_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_gb_reg))
gb_r2 = r2_score(y_test_reg, y_pred_gb_reg)
print(f"Regression - RMSE: {gb_rmse:.4f}, R²: {gb_r2:.4f}")

# Classification
gb_clf = GradientBoostingClassifier(n_estimators=150, learning_rate=0.05,
                                     max_depth=5, subsample=0.8, random_state=42)
gb_clf.fit(X_train, y_train_clf)
y_pred_gb_clf = gb_clf.predict(X_test)
gb_acc = accuracy_score(y_test_clf, y_pred_gb_clf)
print(f"Classification - Accuracy: {gb_acc:.4f}")


TRAINING GRADIENT BOOSTING
Regression - RMSE: 0.2948, R²: 0.9767
Classification - Accuracy: 0.3335


In [ ]:
print("\n" + "="*50)
print("TRAINING XGBOOST")
print("="*50)

# Regression
xgb_reg = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                            subsample=0.8, colsample_bytree=0.8,
                            reg_alpha=0.1, reg_lambda=1.0,
                            random_state=42, tree_method='hist', verbosity=0)
xgb_reg.fit(X_train, y_train_reg)
y_pred_xgb_reg = xgb_reg.predict(X_test)

xgb_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_xgb_reg))
xgb_r2 = r2_score(y_test_reg, y_pred_xgb_reg)
print(f"Regression - RMSE: {xgb_rmse:.4f}, R²: {xgb_r2:.4f}")

# Classification
xgb_clf = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                            subsample=0.8, colsample_bytree=0.8,
                            use_label_encoder=False, eval_metric='mlogloss',
                            random_state=42, tree_method='hist', verbosity=0)
xgb_clf.fit(X_train, y_train_clf)
y_pred_xgb_clf = xgb_clf.predict(X_test)
xgb_acc = accuracy_score(y_test_clf, y_pred_xgb_clf)
print(f"Classification - Accuracy: {xgb_acc:.4f}")



TRAINING XGBOOST
Regression - RMSE: 0.2909, R²: 0.9773
Classification - Accuracy: 0.3337


In [ ]:
print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)

results = pd.DataFrame({
    'Model': ['Random Forest', 'Gradient Boosting', 'XGBoost'],
    'RMSE': [rf_rmse, gb_rmse, xgb_rmse],
    'R2': [rf_r2, gb_r2, xgb_r2],
    'Accuracy': [rf_acc, gb_acc, xgb_acc]
})
print(results.round(4).to_string(index=False))


MODEL COMPARISON
            Model   RMSE     R2  Accuracy
    Random Forest 0.2996 0.9759    0.3324
Gradient Boosting 0.2948 0.9767    0.3335
          XGBoost 0.2909 0.9773    0.3337


In [ ]:

import os

artifacts = {
    'xgb_regressor': xgb_reg,
    'xgb_classifier': xgb_clf,
    'rf_regressor': rf_reg,
    'gb_regressor': gb_reg,
    'label_encoders': le_dict,
    'feature_cols': feature_cols,
    'adoption_labels': adoption_labels.tolist(),
}

with open('corporate_adoption_models.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print(f"\n✅ Models saved to corporate_adoption_models.pkl")
print(f"File size: {os.path.getsize('corporate_adoption_models.pkl') / 1e6:.1f} MB")




✅ Models saved to corporate_adoption_models.pkl
File size: 140.2 MB
